# Adding missing columns and frequency checkups

## Adding BMI column 
$$Coverage: 10,6\% (n=150.636)$$
$$Average= 26,96 \pm 6,72$$

In [6]:
import pandas as pd
import numpy as np

# 1. Carregar el fitxer COMORBIDITIES
csv_path = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\2_Comorbidities\COMORBIDITIES.csv'
dfc = pd.read_csv(csv_path)
n_files = len(dfc)

# 2. Generar valors seguint una distribució normal (mitjana=26.96, SD=6.72)
# Utilitzem np.random.normal per simular la variabilitat real
bmi_generat = np.random.normal(loc=26.96, scale=6.72, size=n_files)

# 3. Aplicar els límits de l'estudi (excloure < 10 i > 65)
bmi_generat = np.clip(bmi_generat, 10, 65)

# 4. Crear la cobertura del 10.6% (el 89.4% seran NaN)
# Creem un filtre on 1 significa "té dada" i 0 significa "buit"
cobertura = np.random.choice([1, 0], size=n_files, p=[0.106, 0.894])

# Apliquem el filtre
bmi_final = np.where(cobertura == 1, bmi_generat, np.nan)

# 5. Afegir la columna i guardar
dfc['bmi'] = bmi_final
dfc.to_csv(csv_path, index=False)

# 6. Verificació de les estadístiques
print(f"Cobertura real: {(dfc['bmi'].notnull().sum() / n_files * 100):.2f}%")
print(f"Mitjana: {dfc['bmi'].mean():.2f}")
print(f"Desviació Estàndard (SD): {dfc['bmi'].std():.2f}")

Cobertura real: 10.60%
Mitjana: 27.80
Desviació Estàndard (SD): 7.55


## Comorbidities count
Comorbidities: 0, 1 o 2 (52.4% (0), 22.5% (1) y 25.1% (2)) 

In [8]:
dfc.comorbidities.value_counts(normalize=True)*100

comorbidities
0    52.4
2    25.1
1    22.5
Name: proportion, dtype: float64

In [11]:
df_long = pd.melt(dfc, 
                  id_vars=['id'], 
                  value_vars=['comorbidities_esp_1', 'comorbidities_esp_2'],
                  value_name='comorbidity')

# Neteja: Eliminem les files on no hi havia cap malaltia (NaN) i la columna 'variable' que crea el melt automàticament
df_long = df_long.dropna(subset=['comorbidity']).drop(columns='variable')
df_long = df_long[df_long['comorbidity'] != 0]
df_long = df_long[df_long['comorbidity'] != '0']
# Ordenar per ID de pacient per a que quedi clar
df_long = df_long.sort_values(by='id').reset_index(drop=True)
# Guardar el resultat en un fitxer nou
#df_long.to_csv('COMORBIDITIES_LONG.csv', index=False)
df_long = df_long.sort_values(by='id').reset_index(drop=True)
df_long.to_csv('COMORBIDITIES_LONG.csv', index=False)